# Hy-MT2-30B-A3B-uncensored-heretic — Colab L4 一鍵自動化翻譯（EN -> 台繁成人向）

**使用步驟：**
1. 確認右上角硬體加速器為 **L4 GPU**（24 GB VRAM；若配到 T4 會自動拒跑並釋放資源）。
2. 確認本機 Agent 已透過 `drive_bridge.py push` 將 `<job>.jsonl` 與 `.manifest.json` 推入 `MyDrive/galgame_mtl/hymt2/input/`。
3. 按 **`Ctrl+F9`（全部執行）**，點「允許」授權 Google Drive 後即可離開電腦。
4. 翻譯完成（或遇錯停止）後，`finally` 區塊會自動保存日誌、`flush_and_unmount()` 同步雲端，並執行 `runtime.unassign()` 釋放 L4 停止計費。

In [ ]:
#@title 1. 執行參數 { display-mode: "form" }
AUTO_DISCONNECT = True #@param {type:"boolean"}
TW_CONVERT_MODE = "s2twp" #@param ["s2twp", "s2t"]
FORCE_RERUN = False #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/galgame_mtl" #@param {type:"string"}
REPO_URL = "https://github.com/poynt2005/hanhua-mtl.git" #@param {type:"string"}
BRANCH = "hymt2-heretic-colab" #@param {type:"string"}

In [ ]:
#@title 2. 一鍵啟動 GPU 4 道驗活 + 4 路並行翻譯 + s2twp QA + 自動斷線
from collections import deque
import shutil
import subprocess
import sys
from pathlib import Path
from google.colab import drive, runtime

def _load_runner_dir(drive_root: Path) -> Path:
    drive_scripts = drive_root / "scripts" / "hymt2"
    if (drive_scripts / "runner.py").is_file() and (drive_scripts / "schema.py").is_file():
        local_scripts = Path("/content/_runner_hymt2")
        local_scripts.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(drive_scripts / "schema.py", local_scripts / "schema.py")
        shutil.copyfile(drive_scripts / "runner.py", local_scripts / "runner.py")
        print(f"[init] 已從 Drive 載入腳本: {drive_scripts}", flush=True)
        return local_scripts

    repo_dir = Path("/content/_repo")
    if repo_dir.exists():
        shutil.rmtree(repo_dir, ignore_errors=True)
    print(f"[init] Drive 無同步腳本，改由 Git 取得 {BRANCH}...", flush=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)],
        check=True,
    )
    return repo_dir / "deploy" / "colab-hymt2"

mounted = False
rc = 1
try:
    drive.mount("/content/drive", force_remount=False)
    mounted = True
    pip_res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "opencc-python-reimplemented"],
        capture_output=True,
        text=True,
    )
    if pip_res.returncode != 0:
        raise RuntimeError(f"pip install opencc failed:\n{pip_res.stdout}\n{pip_res.stderr}")
    runner_dir = _load_runner_dir(Path(DRIVE_ROOT))
    work_dir = Path(DRIVE_ROOT) / "hymt2" / "work"
    work_dir.mkdir(parents=True, exist_ok=True)
    runner_log = work_dir / "runner.log"
    cmd = [
        sys.executable,
        "-u",
        str(runner_dir / "runner.py"),
        "--drive-root", DRIVE_ROOT,
        "--tw-mode", TW_CONVERT_MODE,
    ]
    if FORCE_RERUN:
        cmd.append("--force")
    tail = deque(maxlen=30)
    with open(runner_log, "w", encoding="utf-8", newline="\n") as log_fh:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            log_fh.write(line)
            log_fh.flush()
            tail.append(line.rstrip("\r\n"))
        rc = proc.wait()
    if rc != 0:
        tail_txt = "\n".join(tail)
        raise RuntimeError(
            f"runner.py 結束碼 {rc} (完整日誌已存至 hymt2/work/runner.log)。最後輸出：\n{tail_txt}"
        )
    print("\n[DONE] 全部翻譯任務已完成並寫入 .DONE.json！", flush=True)
finally:
    if mounted:
        try:
            print("[flush] 正在將輸出與日誌同步回 Google Drive...")
            drive.flush_and_unmount()
            print("[flush] Drive 同步完成。")
        except Exception as exc:
            print(f"[flush] warn: {exc}")
    if AUTO_DISCONNECT:
        print("[power] AUTO_DISCONNECT=True -> 正在釋放 Colab GPU 執行階段...")
        runtime.unassign()
